# CTB ProSiT — Load Saved Parameters and Simulate

This notebook loads the from the project generated saved simulation parameters from the thesis and runs ProSiT's simulator on them.
The workflow follows the ProSiT documentation (https://github.com/franvinci/prosit#save-and-load-parameters):

1. Load the Petri net (PNML)
2. Load saved parameters (pickle, since the ProSiT JSON loader has a known limitation for this model)
3. Create a `SimulatorEngine` and call `.apply()`
4. Inspect the output

No event log or discovery step is needed for the requested reproduction — the parameters are already discovered, saved and provided in /models .

In [ ]:
%pip install -r requirements.txt --disable-pip-version-check -q

## 1. Load Petri net and saved parameters

In [ ]:
import pickle
from datetime import datetime
from copy import deepcopy

import pm4py
from prosit import SimulatorParameters, SimulatorEngine

# Load the common Petri net
net, im, fm = pm4py.read_pnml("models/ctb_inductive_miner.pnml")
print(f"Petri net: {len(net.places)} places, {len(net.transitions)} transitions, {len(net.arcs)} arcs")

# Load saved parameter bundles
with open("models/params_baseline_rmg_max_concurrency_3.pkl", "rb") as f:
    baseline = pickle.load(f)

with open("models/params_rules_only_workload_blind.pkl", "rb") as f:
    rules_only = pickle.load(f)

with open("models/params_t22_closed.pkl", "rb") as f:
    t22_closed = pickle.load(f)

with open("models/params_demand_plus_20pct.pkl", "rb") as f:
    demand_plus_20 = pickle.load(f)

## 2. Inspect model configurations

The thesis compares three ProSiT configurations:
- **rules_only**: decision-tree rules enabled, no workload features (intermediate baseline)
- **baseline**: rules + workload features, RMG concurrency capped at 3 (effective baseline for scenarios)
- **t22_closed**: baseline with T22 removed from RMG pools (Scenario A)
- **demand_plus_20**: baseline with 20% higher arrival intensity (Scenario B)

In [ ]:
for name, params in [("rules_only", rules_only), ("baseline", baseline),
                     ("t22_closed", t22_closed), ("demand_plus_20", demand_plus_20)]:
    print(f"\n--- {name} ---")
    print(f"  rules_mode:          {params.rules_mode}")
    print(f"  workload_features:   {params.use_workload_features}")
    print(f"  activities:          {list(params.act_to_resources.keys())}")
    print(f"  RMG resources:       {len(params.act_to_resources['RMG_receive'])}")
    print(f"  T22 in RMG_receive:  {'T22' in params.act_to_resources['RMG_receive']}")
    print(f"  max_concurrency T06: {params.max_concurrency.get('T06', '—')}")

## 3. Run ProSiT's simulator directly

This is the standard ProSiT usage from the README:
```python
engine = SimulatorEngine(params)
sim_log = engine.apply(n_traces=..., t_start=...)
```

I simulate 1000 cases from the holdout start time for a quick check.

In [ ]:
import random
random.seed(42)

t_start = datetime(2026, 4, 20, 18, 17)  # holdout split boundary

engine = SimulatorEngine(deepcopy(baseline))
sim_log = engine.apply(n_traces=1000, t_start=t_start)

print(f"Simulated {sim_log['case:concept:name'].nunique()} cases, {len(sim_log)} events")
print(f"Columns: {sim_log.columns.tolist()}")
sim_log.head(10)

## 4. Basic KPI comparison across scenarios

Run 1000 cases for each scenario and compare mean turnaround time.

In [ ]:
import pandas as pd

results = []
for name, params in [("baseline", baseline), ("t22_closed", t22_closed), ("demand_plus_20", demand_plus_20)]:
    random.seed(42)
    engine = SimulatorEngine(deepcopy(params))
    log = engine.apply(n_traces=1000, t_start=t_start)

    # turnaround = last event time - first event time per case
    log["time:timestamp"] = pd.to_datetime(log["time:timestamp"])
    case_times = log.groupby("case:concept:name")["time:timestamp"]
    turnaround = (case_times.max() - case_times.min()).dt.total_seconds() / 60
    results.append({"scenario": name, "mean_turnaround_min": turnaround.mean(),
                    "median_turnaround_min": turnaround.median(), "cases": log["case:concept:name"].nunique()})

pd.DataFrame(results)

## 4b. Rules-only intermediate (Table 4.1 configuration)

The rules-only model uses `max_depth_tree=3` but `use_workload_features=False` and removes
all workload-proxy attributes before discovery. This isolates the effect of decision rules
without workload conditioning (thesis Section 4.5).

In [ ]:
import json

with open("models/rules_only_workload_blind_run_summary.json") as f:
    summary = json.load(f)

print("Discovery settings for rules-only intermediate:")
print(f"  max_depth_tree:         {summary['hyperparameters']['max_depth_tree']}")
print(f"  use_workload_features:  {summary['hyperparameters']['use_workload_features']}")
print(f"  workload_blind:         {summary['hyperparameters']['workload_blind_attributes']}")
print(f"  attribute_mode:         {summary['hyperparameters']['attribute_mode']}")
print(f"\nRemoved workload-proxy attributes ({len(summary['hyperparameters']['workload_proxy_attributes_removed'])}):")
for attr in summary["hyperparameters"]["workload_proxy_attributes_removed"]:
    print(f"    {attr}")

print(f"\nConformance on held-out test:")
for row in summary["conformance"]:
    if row["log"] == "test":
        print(f"  fitness={row['fitness_log']:.6f}  precision={row['precision_token']:.4f}")

# Quick single-seed comparison
random.seed(42)
engine = SimulatorEngine(deepcopy(rules_only))
log_ro = engine.apply(n_traces=1000, t_start=t_start)
log_ro["time:timestamp"] = pd.to_datetime(log_ro["time:timestamp"])
case_t = log_ro.groupby("case:concept:name")["time:timestamp"]
tt = (case_t.max() - case_t.min()).dt.total_seconds() / 60
print(f"\nRules-only quick run (1000 cases, seed 42): mean turnaround = {tt.mean():.1f} min")

## 5. ProSiT JSON export/import demonstration

ProSiT provides `to_json()` / `from_json()` for portable parameter saving.
For this CTB model, `from_json()` cannot fully restore the empirical arrival samples (known ProSiT 1.0.3 limitation with `nan` attribute keys). The pickle is therefore the authoritative saved state.

In [ ]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)

# Export (works)
baseline.to_json("outputs/baseline_params.json")
print("Exported to outputs/baseline_params.json")

# Import attempt (shows the known limitation)
params_reloaded = SimulatorParameters(net, im, fm)
try:
    params_reloaded.from_json("outputs/baseline_params.json")
    print("JSON reload: success")
except Exception as e:
    print(f"JSON reload failed (expected): {type(e).__name__}: {e}")
    print("This is why the thesis uses verified pickle files for exact reproduction.")

## 6. Full thesis reproduction (~30 min)

This runs the exact 10-seed × 3-scenario × 17,892-case experiment from Tables 5.3 and 5.11.
The final cell compares each generated result table against the frozen thesis outputs.

In [ ]:
import sys
sys.path.insert(0, ".")
import reviewer_runner as rr

rr.verify_package_files()
output_dir = rr.run_saved_models(mode="full")

comparison = rr.compare_with_frozen_results(output_dir)
print(comparison.to_string(index=False))
print("\nFULL REPRODUCTION PASS — all thesis KPIs matched.")

In [ ]:
import numpy as np

kpi = pd.read_csv(output_dir / "scenario_kpi_summary.csv")
deltas = pd.read_csv(output_dir / "scenario_paired_delta_summary.csv")

print("=== Thesis Table 5.11: Scenario KPIs (mean ± 95% CI) ===\n")
for scenario in ["baseline", "t22_closed", "demand_plus_20pct"]:
    row = kpi[(kpi["scenario"] == scenario) & (kpi["metric"] == "mean_turnaround_min")].iloc[0]
    print(f"  {scenario:20s}  mean_turnaround = {row['mean']:.3f} [{row['ci95_lo']:.3f}, {row['ci95_hi']:.3f}]")

print("\n=== Thesis Table 5.11: Paired deltas (scenario − baseline) ===\n")
for _, row in deltas[deltas["metric"] == "mean_turnaround_min"].iterrows():
    print(f"  {row['scenario']:20s}  Δ = {row['mean_delta']:+.3f} [{row['ci95_delta_lo']:+.3f}, {row['ci95_delta_hi']:+.3f}]")